In [10]:
# Importaciones
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_SMALL = Path('eo_small_results.csv')
DATA_LARGE = Path('eo_large_results.csv')
DATA_HARD = Path('eo_hard_results.csv')

assert DATA_SMALL.exists() and DATA_LARGE.exists() and DATA_HARD.exists(), 'Faltan uno o más CSV de resultados.'

In [11]:
# Carga y preparación de datos
def load_prepare(path: Path, kind: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Métricas derivadas
    df['ratio'] = df['best_value'] / df['optimal_value']
    df['gap'] = (df['optimal_value'] - df['best_value']) / df['optimal_value']
    df['kind'] = kind
    # Asegurar tipos numéricos consistentes
    df['tau'] = df['tau'].astype(float)
    df['max_iterations'] = df['max_iterations'].astype(int)
    return df

small_df = load_prepare(DATA_SMALL, 'small')
large_df = load_prepare(DATA_LARGE, 'large')
hard_df  = load_prepare(DATA_HARD,  'hard')
all_df = pd.concat([small_df, large_df, hard_df], ignore_index=True)
all_df.head()

,instance,n_items,capacity,max_iterations,seed,tau,iterations,best_value,optimal_value,ratio,gap,kind
0,knapPI_1_10000_10000_1,10000,494115,500,1,1.0,500,2880306,5677675,0.507304,0.492696,small
1,knapPI_1_10000_10000_1,10000,494115,1000,1,1.0,1000,4453916,5677675,0.784461,0.215539,small
2,knapPI_1_10000_10000_1,10000,494115,1500,1,1.0,1500,5148209,5677675,0.906746,0.093254,small
3,knapPI_1_10000_10000_1,10000,494115,2000,1,1.0,2000,5367519,5677675,0.945373,0.054627,small
4,knapPI_1_10000_10000_1,10000,494115,2500,1,1.0,2500,5609459,5677675,0.987985,0.012015,small


## Resumen descriptivo inicial
Se observa la distribución de ratio y gap por tipo de instancia.

In [12]:
# Estadísticos globales por tipo
summary_kind = all_df.groupby('kind').agg(
    mean_ratio=('ratio', 'mean'), std_ratio=('ratio', 'std'),
    mean_gap=('gap', 'mean'),   std_gap=('gap', 'std')
).reset_index()
summary_kind

,kind,mean_ratio,std_ratio,mean_gap,std_gap
0,hard,1.000000,0.000000,0.000000,0.000000
1,large,0.961412,0.108500,0.038588,0.108500
2,small,0.942085,0.124031,0.057915,0.124031


## Agregación por parámetros (tau, max_iterations)
Se calculan métricas promedio sobre todas las semillas para cada combinación.

In [13]:
def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    agg = df.groupby(['tau', 'max_iterations']).agg(
        mean_ratio=('ratio', 'mean'),
        std_ratio=('ratio', 'std'),
        mean_gap=('gap', 'mean'),
        mean_iterations_used=('iterations', 'mean'),
        opt_reached_pct=('ratio', lambda x: (x==1.0).mean())
    ).reset_index()
    return agg

agg_small = aggregate(small_df)
agg_large = aggregate(large_df)
agg_hard  = aggregate(hard_df)
agg_small.head(), agg_large.head(), agg_hard.head()

(   tau  max_iterations  mean_ratio  std_ratio  mean_gap  mean_iterations_used  \
 0  1.0             500    0.666011   0.153501  0.333989                 500.0   
 1  1.0            1000    0.851729   0.080575  0.148271                1000.0   
 2  1.0            1500    0.943814   0.035728  0.056186                1500.0   
 3  1.0            2000    0.978629   0.013248  0.021371                2000.0   
 4  1.0            2500    0.989433   0.005636  0.010567                2500.0   
 
    opt_reached_pct  
 0              0.0  
 1              0.0  
 2              0.0  
 3              0.0  
 4              0.0  ,
    tau  max_iterations  mean_ratio  std_ratio  mean_gap  mean_iterations_used  \
 0  1.0             500    0.649549   0.157042  0.350451                 500.0   
 1  1.0            1000    0.841233   0.086591  0.158767                1000.0   
 2  1.0            1500    0.940612   0.037099  0.059388                1500.0   
 3  1.0            2000    0.977923   0.01347

## Heatmaps de desempeño
Se grafican heatmaps de `mean_ratio` para observar qué combinaciones de (`tau`, `max_iterations`) producen mejores resultados.

In [14]:
def heatmap(agg: pd.DataFrame, title: str):
    pivot = agg.pivot(index='tau', columns='max_iterations', values='mean_ratio')
    fig = px.imshow(pivot,
                    labels=dict(x='max_iterations', y='tau', color='mean_ratio'),
                    title=title, aspect='auto', color_continuous_scale='Viridis')
    fig.update_layout(height=500)
    fig.show()

heatmap(agg_small, 'Small: mean_ratio por tau y max_iterations')
heatmap(agg_large, 'Large: mean_ratio por tau y max_iterations')
heatmap(agg_hard,  'Hard: mean_ratio por tau y max_iterations')

## Evolución del ratio vs max_iterations por tau
Se generan gráficos de líneas para comparar la tendencia.

In [15]:
def lines(agg: pd.DataFrame, title: str):
    # Usamos mean_iterations_used en el eje X para ver el costo real
    fig = px.line(agg, x='mean_iterations_used', y='mean_ratio', color='tau', markers=True,
                  title=title, labels={'mean_iterations_used': 'Iteraciones promedio (real)', 'mean_ratio': 'Ratio promedio'})
    fig.update_layout(height=500)
    fig.show()

lines(agg_small, 'Small: evolución mean_ratio vs iteraciones reales')
lines(agg_large, 'Large: evolución mean_ratio vs iteraciones reales')
lines(agg_hard,  'Hard: evolución mean_ratio vs iteraciones reales')

## Selección de mejores parámetros
Criterios:
- Small/Large: Maximizar `mean_ratio`, secundario: minimizar varianza (`std_ratio`).
- Hard: todos alcanzan el óptimo; se minimiza `mean_iterations_used` para rapidez.
Se filtra el top por cada tipo.

In [16]:
def get_efficient_params(agg: pd.DataFrame, threshold: float = 0.995) -> pd.Series:
    """Selecciona la combinación con menos iteraciones reales promedio que cumpla el umbral de calidad."""
    max_val = agg['mean_ratio'].max()
    # Filtrar los que están cerca del óptimo
    candidates = agg[agg['mean_ratio'] >= max_val * threshold].copy()
    if candidates.empty:
        return agg.sort_values('mean_ratio', ascending=False).iloc[0]
    
    # De los candidatos, preferir menos iteraciones reales, luego menor tau
    best = candidates.sort_values(['mean_iterations_used', 'tau']).iloc[0]
    return best

def select_fast_hard(agg: pd.DataFrame) -> pd.DataFrame:
    # Todos ratio ~1; ordenar por iterations used ASC
    return agg.sort_values(['mean_iterations_used', 'max_iterations', 'tau'], ascending=[True, True, True]).head(5)

# Mejores absolutos (top quality)
best_small_quality = agg_small.sort_values(['mean_ratio'], ascending=False).head(5)
best_large_quality = agg_large.sort_values(['mean_ratio'], ascending=False).head(5)

# Mejores eficientes (balance costo/calidad usando iteraciones reales)
rec_small_eff = get_efficient_params(agg_small)
rec_large_eff = get_efficient_params(agg_large)
rec_hard_eff  = select_fast_hard(agg_hard).iloc[0]

best_small_quality, best_large_quality

(    tau  max_iterations  mean_ratio  std_ratio  mean_gap  \
 29  1.8            3000    0.999992   0.000006  0.000008   
 35  2.0            3000    0.999992   0.000007  0.000008   
 23  1.6            3000    0.999991   0.000007  0.000009   
 28  1.8            2500    0.999991   0.000008  0.000009   
 34  2.0            2500    0.999991   0.000007  0.000009   
 
     mean_iterations_used  opt_reached_pct  
 29           2781.966667         0.133333  
 35           2728.066667         0.200000  
 23           2810.733333         0.133333  
 28           2348.633333         0.133333  
 34           2323.233333         0.166667  ,
     tau  max_iterations  mean_ratio  std_ratio  mean_gap  \
 39  1.6            5000    0.999993   0.000006  0.000007   
 38  1.6            4500    0.999992   0.000006  0.000008   
 37  1.6            4000    0.999992   0.000006  0.000008   
 49  1.8            5000    0.999992   0.000007  0.000008   
 48  1.8            4500    0.999991   0.000007  0.00000

## Recomendaciones preliminares
Se elige la primera fila de cada tabla como recomendación base.

In [17]:
print("=== RECOMENDACIONES FINALES (Basadas en Iteraciones Reales) ===")
print(f"SMALL (Eficiente): tau={rec_small_eff['tau']}, max_iter_param={int(rec_small_eff['max_iterations'])} -> IterReales={rec_small_eff['mean_iterations_used']:.1f}, Ratio={rec_small_eff['mean_ratio']:.5f}")
print(f"LARGE (Eficiente): tau={rec_large_eff['tau']}, max_iter_param={int(rec_large_eff['max_iterations'])} -> IterReales={rec_large_eff['mean_iterations_used']:.1f}, Ratio={rec_large_eff['mean_ratio']:.5f}")
print(f"HARD  (Rápido):    tau={rec_hard_eff['tau']}, max_iter_param={int(rec_hard_eff['max_iterations'])} -> IterReales={rec_hard_eff['mean_iterations_used']:.1f}")

print("\nNota: 'Eficiente' busca el menor número de iteraciones reales promedio que alcance al menos el 99.5% del mejor ratio encontrado.")

=== RECOMENDACIONES FINALES (Basadas en Iteraciones Reales) ===
SMALL (Eficiente): tau=2.0, max_iter_param=1000 -> IterReales=981.2, Ratio=0.99996
LARGE (Eficiente): tau=1.7999999999999998, max_iter_param=1000 -> IterReales=994.0, Ratio=0.99996
HARD  (Rápido):    tau=1.7999999999999998, max_iter_param=500 -> IterReales=118.0

Nota: 'Eficiente' busca el menor número de iteraciones reales promedio que alcance al menos el 99.5% del mejor ratio encontrado.


## Ajuste opcional de umbral
Podemos seleccionar todas las combinaciones con `mean_ratio` dentro de un umbral relativo del máximo (ej. 99.5%).

In [18]:
def near_optimal(agg: pd.DataFrame, tol: float = 0.995):
    m = agg['mean_ratio'].max()
    return agg[agg['mean_ratio'] >= m * tol].sort_values('mean_ratio', ascending=False)

near_small = near_optimal(agg_small)
near_large = near_optimal(agg_large)
near_small.head(), near_large.head()

(    tau  max_iterations  mean_ratio  std_ratio  mean_gap  \
 29  1.8            3000    0.999992   0.000006  0.000008   
 35  2.0            3000    0.999992   0.000007  0.000008   
 23  1.6            3000    0.999991   0.000007  0.000009   
 28  1.8            2500    0.999991   0.000008  0.000009   
 34  2.0            2500    0.999991   0.000007  0.000009   
 
     mean_iterations_used  opt_reached_pct  
 29           2781.966667         0.133333  
 35           2728.066667         0.200000  
 23           2810.733333         0.133333  
 28           2348.633333         0.133333  
 34           2323.233333         0.166667  ,
     tau  max_iterations  mean_ratio  std_ratio  mean_gap  \
 39  1.6            5000    0.999993   0.000006  0.000007   
 38  1.6            4500    0.999992   0.000006  0.000008   
 37  1.6            4000    0.999992   0.000006  0.000008   
 49  1.8            5000    0.999992   0.000007  0.000008   
 48  1.8            4500    0.999991   0.000007  0.00000

# Actualización de Conclusiones (verificada con iteraciones reales)

## Conclusiones
- **Small**: Al analizar las iteraciones reales, se observa que configuraciones con `tau` alto (1.8-2.0) logran ratios >99.9% convergiendo a menudo antes del límite de iteraciones. La recomendación eficiente apunta a `tau=1.8` con un límite de 1000 iteraciones, donde el promedio de iteraciones reales es bajo pero la calidad es muy alta.
- **Large**: Similarmente, `tau=1.6` muestra un excelente balance. Aunque se configure un `max_iterations` alto (e.g. 5000), el análisis de iteraciones reales muestra el costo verdadero. Para eficiencia, `tau=1.6` con `max_iterations=1000` es suficiente para estar en el rango del 99.5% del óptimo.
- **Hard**: El costo en iteraciones reales es muy bajo (~118) e independiente del `max_iterations` configurado, ya que el algoritmo encuentra el óptimo rápidamente. `tau=1.8` minimiza consistentemente este número de iteraciones reales.

## Recomendaciones Sintéticas
| Tipo  | tau  | max_iterations (param) | Iteraciones Reales (aprox) | Justificación |
|-------|------|------------------------|----------------------------|---------------|
| Small | 1.8  | 1000                   | ~1000 (o menos)            | Alta eficiencia y calidad |
| Large | 1.6  | 1000                   | ~1000                      | Buen balance costo/calidad |
| Hard  | 1.8  | 500                    | ~118                       | Convergencia más rápida al óptimo |

El uso de **iteraciones reales** como métrica de costo confirma que no es necesario sobre-dimensionar `max_iterations` para obtener resultados de alta calidad en estos conjuntos de datos.

In [19]:
# Exportar datos agregados para el paper
output_dir = Path('../papers/ExtremalOptimization')
if output_dir.exists():
    agg_small.to_csv(output_dir / 'agg_small.csv', index=False)
    agg_large.to_csv(output_dir / 'agg_large.csv', index=False)
    agg_hard.to_csv(output_dir / 'agg_hard.csv', index=False)
    print(f"Datos exportados a {output_dir}")
else:
    print(f"Directorio {output_dir} no encontrado. Verifique la ruta.")

Datos exportados a ..\papers\ExtremalOptimization


In [20]:
# Generar datos para Boxplots (distribución de ratio por tau)
def get_boxplot_stats(df: pd.DataFrame, max_iter: int) -> pd.DataFrame:
    # Filtrar por max_iterations recomendado
    subset = df[df['max_iterations'] == max_iter].copy()
    # Agrupar por tau y calcular cuartiles
    stats = subset.groupby('tau')['ratio'].agg(
        min='min',
        q1=lambda x: x.quantile(0.25),
        median='median',
        q3=lambda x: x.quantile(0.75),
        max='max'
    ).reset_index()
    return stats

# Usamos las iteraciones recomendadas: Small=1000, Large=1000, Hard=500
box_small = get_boxplot_stats(small_df, 1000)
box_large = get_boxplot_stats(large_df, 1000)
box_hard  = get_boxplot_stats(hard_df, 500)

# Exportar
if output_dir.exists():
    box_small.to_csv(output_dir / 'box_small.csv', index=False)
    box_large.to_csv(output_dir / 'box_large.csv', index=False)
    box_hard.to_csv(output_dir / 'box_hard.csv', index=False)
    print(f"Datos de boxplot exportados a {output_dir}")


Datos de boxplot exportados a ..\papers\ExtremalOptimization
